# Remove Bad Tissue

This notebook applies the Labelme polygons saved in each sample's `filtering/topic_image.json` file to the step-02 initial-neighborhood AnnData objects.

It uses the matching helper script at `~/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/Aaron/Xenium-functions-from-Alex/03_Remove_Bad_Tissue.py`, writes filtered AnnData files to the output directory you choose below, and leaves a `bad_tissue_removed_adatas` dictionary available at the end.

## Chunk 1: Imports and Helper Module

Run this first so the notebook can call the same functions used by the command-line step-03 script.

In [1]:

from argparse import Namespace
import importlib.util
import json
from pathlib import Path
import sys

import pandas as pd
import scanpy as sc

PROCESSING_DIR = Path("~/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/Aaron/Xenium-functions-from-Alex").expanduser()
HELPER_PATH = PROCESSING_DIR / "03_Remove_Bad_Tissue.py"

spec = importlib.util.spec_from_file_location("remove_bad_tissue", HELPER_PATH)
remove_bad_tissue = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = remove_bad_tissue
spec.loader.exec_module(remove_bad_tissue)

HELPER_PATH

PosixPath('/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/Aaron/Xenium-functions-from-Alex/03_Remove_Bad_Tissue.py')

## Chunk 2: Parameters

Set `OUTPUT_ROOT` to the folder where the bad-tissue-removed results should be written.

In [4]:
BASE_DIR = Path('/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/RESULTS/20240627__192310__KAECH_AD_GBM_240627').expanduser()

INPUT_ROOT = BASE_DIR / "02_Calculating_initial_neighborhood"
OUTPUT_ROOT = BASE_DIR / "03_Remove_bad_tissues"
INPUT_NAME = "02_initial_neighborhoods.h5ad"

SAMPLES = ["AD_inf", "AD_mock", "reference_ln"]
INPUT_NAME = "02_initial_neighborhoods.h5ad"
OUTPUT_NAME = "03_tissue_filtered.h5ad"
ANNOTATION_NAME = "topic_image.json"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT

PosixPath('/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/RESULTS/20240627__192310__KAECH_AD_GBM_240627/03_Remove_bad_tissues')

## Chunk 3: Validate Inputs

This checks that every sample has the step-02 AnnData file and the matching Labelme JSON.

In [5]:
# prefer JSON; fall back to PNG if JSON missing
sample_dirs = {sample: INPUT_ROOT / sample for sample in SAMPLES}
input_paths = {
    sample: sample_dir / "adatas" / INPUT_NAME
    for sample, sample_dir in sample_dirs.items()
}
annotation_paths = {}
for sample, sample_dir in sample_dirs.items():
    json_path = sample_dir / "filtering" / "topic_image.json"
    png_path = sample_dir / "filtering" / "topic_image.png"
    annotation_paths[sample] = json_path if json_path.exists() else png_path

missing_inputs = {sample: path for sample, path in input_paths.items() if not path.exists()}
missing_annotations = {sample: path for sample, path in annotation_paths.items() if not path.exists()}

if missing_inputs or missing_annotations:
    lines = []
    if missing_inputs:
        lines.append("Missing step-02 AnnData files:")
        lines.extend(f"- {sample}: {path}" for sample, path in missing_inputs.items())
    if missing_annotations:
        lines.append("Missing Labelme JSON/PNG files:")
        lines.extend(f"- {sample}: {path}" for sample, path in missing_annotations.items())
    raise FileNotFoundError("\n".join(lines))

## Chunk 3: Validate Inputs

This checks that every sample has the step-02 AnnData file and the matching Labelme JSON.

In [6]:
sample_dirs = {sample: INPUT_ROOT / sample for sample in SAMPLES}
input_paths = {
    sample: sample_dir / "adatas" / INPUT_NAME
    for sample, sample_dir in sample_dirs.items()
}
annotation_paths = {
    sample: sample_dir / "filtering" / ANNOTATION_NAME
    for sample, sample_dir in sample_dirs.items()
}

missing_inputs = {sample: path for sample, path in input_paths.items() if not path.exists()}
missing_annotations = {
    sample: path for sample, path in annotation_paths.items() if not path.exists()
}

if missing_inputs or missing_annotations:
    lines = []
    if missing_inputs:
        lines.append("Missing step-02 AnnData files:")
        lines.extend(f"- {sample}: {path}" for sample, path in missing_inputs.items())
    if missing_annotations:
        lines.append("Missing Labelme JSON files:")
        lines.extend(f"- {sample}: {path}" for sample, path in missing_annotations.items())
    raise FileNotFoundError("\n".join(lines))

pd.DataFrame(
    [
        {
            "sample": sample,
            "input_adata": input_paths[sample],
            "annotation_json": annotation_paths[sample],
            "output_adata": OUTPUT_ROOT / sample / "adatas" / OUTPUT_NAME,
        }
        for sample in SAMPLES
    ]
)

,sample,input_adata,annotation_json,output_adata
0,AD_inf,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...
1,AD_mock,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...
2,reference_ln,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...


## Chunk 4: Preview Annotations
Use this quick check to confirm the JSON files contain the polygons you draw

In [7]:
annotation_summary = []
for sample, annotation_path in annotation_paths.items():
    annotation = json.loads(annotation_path.read_text())
    shapes = annotation.get("shapes", [])
    labels = sorted({str(shape.get("label", "")) for shape in shapes})
    annotation_summary.append(
        {
            "sample": sample,
            "num_shapes": len(shapes),
            "labels": labels,
            "image_path": annotation.get("imagePath"),
            "annotation_json": annotation_path
        }
    )
    pd.DataFrame(annotation_summary)

## Chunk 5: Run Bad-Tissue Removal
This cell writes one filtered AnnData object per sample, plus summary files and a post-removal topic image for checking

In [8]:
args = Namespace(
    input_name=INPUT_NAME,
    output_name=OUTPUT_NAME,
    output_root = str(OUTPUT_ROOT),
    annotation_name = ANNOTATION_NAME,
    annotation_json = None,
    removal_label=None
)
results = {}
for sample in SAMPLES:
    result = remove_bad_tissue.process_experiment(sample_dirs[sample], args)
    results['sample']  = sample
    results['sample'] = result

list(results)

Processing /Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/RESULTS/20240627__192310__KAECH_AD_GBM_240627/02_Calculating_initial_neighborhood/AD_inf
Removed 2049 / 57466 cells across 2 polygons
Wrote /Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/RESULTS/20240627__192310__KAECH_AD_GBM_240627/03_Remove_bad_tissues/AD_inf/filtering/topic_image_after_removal.png
Wrote /Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/RESULTS/20240627__192310__KAECH_AD_GBM_240627/03_Remove_bad_tissues/AD_inf/adatas/03_tissue_filtered.h5ad
Processing /Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518

['sample']

## Chunk 6: Review Ouputs
The summary table reports how many cells were removed from each sample and where the outputs were written

In [9]:
summary_rows = []
for sample in SAMPLES:
    output_dir = OUTPUT_ROOT / sample
    summary_path = output_dir / "filtering" / "bad_tissue_filter_summary.json"
    polygon_table_path = output_dir / "filtering" / "bad_tissue_filter_polygons.csv"

    if not summary_path.exists():
        raise FileNotFoundError(
            f"Missing summary for {sample}. Run Chunk 5 first: {summary_path}"
        )

    summary = json.loads(summary_path.read_text())
    summary_rows.append(
        {
            "sample": sample,
            "cells_before": summary["num_cells_before"],
            "cells_removed": summary["num_cells_removed"],
            "cells_after": summary["num_cells_after"],
            "fraction_removed": summary["fraction_cells_removed"],
            "polygons": summary["num_annotation_polygons"],
            "output_adata": Path(summary["output_adata"]),
            "filtered_topic_image": Path(summary["filtered_topic_image_path"]),
            "summary_json": summary_path,
            "polygon_csv": polygon_table_path,
        }
    )

removal_summary = pd.DataFrame(summary_rows)
removal_summary

,sample,cells_before,cells_removed,cells_after,fraction_removed,polygons,output_adata,filtered_topic_image,summary_json,polygon_csv
0,AD_inf,57466,2049,55417,0.035656,2,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...
1,AD_mock,64594,217,64377,0.003359,7,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...
2,reference_ln,45469,7151,38318,0.157272,7,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...,/Users/valishashah/Library/CloudStorage/Box-Bo...


## Chunk 7: Final Filtered AnnData Objects

After this cell, `bad_tissue_removed_adatas` contains the filtered AnnData object for each sample.

In [10]:
bad_tissue_removed_paths = {
    sample: OUTPUT_ROOT / sample / "adatas" / OUTPUT_NAME
    for sample in SAMPLES
}

missing_outputs = {
    sample: path for sample, path in bad_tissue_removed_paths.items() if not path.exists()
}
if missing_outputs:
    missing_text = "\n".join(f"- {sample}: {path}" for sample, path in missing_outputs.items())
    raise FileNotFoundError(f"Missing filtered AnnData outputs:\n{missing_text}")

bad_tissue_removed_adatas = {
    sample: sc.read_h5ad(path)
    for sample, path in bad_tissue_removed_paths.items()
}

pd.DataFrame(
    [
        {
            "sample": sample,
            "cells": adata.n_obs,
            "features": adata.n_vars,
            "adata_path": bad_tissue_removed_paths[sample],
        }
        for sample, adata in bad_tissue_removed_adatas.items()
    ]
)

,sample,cells,features,adata_path
0,AD_inf,55417,480,/Users/valishashah/Library/CloudStorage/Box-Bo...
1,AD_mock,64377,480,/Users/valishashah/Library/CloudStorage/Box-Bo...
2,reference_ln,38318,480,/Users/valishashah/Library/CloudStorage/Box-Bo...
